# Queimar Legenda MULTICOR (classificação gramatical por palavra)

Pega um arquivo `.ass` (o que saiu do `caption-multicolor-generate.ipynb` ou do
`caption-multicolor-zh-generate.ipynb` — pode ser o original ou uma versão que
você corrigiu manualmente) e queima no vídeo/imagem de fundo do vídeo.

**Serve as duas variantes de idioma.** Não existe um
`caption-multicolor-zh-burn.ipynb` porque não precisa: o `.ass` de 6 idiomas
sai com `_zh` no nome, e a célula 5 lê esse sufixo pra gravar o resultado em
`<nome>_final_multicolor_zh.mp4` em vez de por cima do de 5 idiomas.

No final, duas ações **separadas**: baixar o vídeo final pra conferir, e
(só depois de confirmar que ficou bom) salvar no Drive.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. SETUP                                                        ║
# ╚══════════════════════════════════════════════════════════════════╗
# Garante a fonte do coreano (Noto Sans CJK) instalada — sem isso, o
# libass (o que desenha a legenda em cima do vídeo) não acha os
# caracteres Hangul e mostra quadradinho (□) no lugar. Não dá pra supor
# que já vem instalada no ambiente do Colab.
!apt-get install -y -qq fonts-noto-cjk > /dev/null 2>&1

import shutil, sys
from pathlib import Path
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

PASTA_DRIVE_RAIZ_MODULOS = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ_MODULOS}/pipeline/modulos")
DESTINO = Path("/content/pipeline")
if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados")

# ── O Drive tem TODOS os módulos? ──────────────────────────────────────────
# "N módulos copiados" sozinho não quer dizer nada: um Drive com 13 dos 31
# saía com visto verde, e o notebook quebrava depois num import, longe da
# causa. O manifesto (gravado pelo repositorio-sincronizar) diz quantos
# DEVIAM estar lá.
_manifesto = PASTA_MODULOS / "_manifesto.txt"
if _manifesto.exists():
    _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                  if l.strip() and not l.startswith("#")}
    _faltando = sorted(_esperados - {f.name for f in DESTINO.glob("*.py")})
    if _faltando:
        print(f"\n🚨 FALTAM {len(_faltando)} de {len(_esperados)} módulos no Drive:")
        for _n in _faltando:
            print(f"     {_n}")
        raise SystemExit(
            "Rode o repositorio-sincronizar.ipynb antes de continuar. "
            "Seguir assim quebra num import lá na frente, longe da causa.")
    print(f"   ✅ os {len(_esperados)} módulos do manifesto estão presentes")
else:
    print("   ⚠️  sem _manifesto.txt — rode o repositorio-sincronizar pra criá-lo")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")
if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

print("✅ Setup concluído")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
NOME_ORACAO = "40_Matt_02"
PASTA_DRIVE_RAIZ = "narrated_video"

from config import PipelineConfig
from drive_utils import DriveClient


# Fundo: "imagem", "video", ou None pra detectar sozinho pelo arquivo que
# existe no Drive. Só preencha à mão se as duas versões estiverem lá.
MODO_CLIPE = None

# O modo do fundo (imagem parada ou clipe de vídeo) sai do arquivo que EXISTE
# no Drive, não de uma opção que dá pra esquecer de marcar -- mesma ideia do
# sufixo `_zh` lido do nome do .ass. Sem isto, queimar sobre um vídeo base
# feito em modo imagem procurava `_video_base.mp4` e não achava; e se achasse,
# o resultado sairia sem `_img`, por cima da versão de clipe.
import config as _cfgmod
if MODO_CLIPE is None:
    MODO_CLIPE = _cfgmod.detectar_modo_clipe(
        Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/videos/{NOME_ORACAO}"),
        NOME_ORACAO,
    )
    print(f"🎞️  Background mode detected from Drive: {MODO_CLIPE}")

config = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ,
                        IDIOMA_MESTRE="en", MODO_CLIPE=MODO_CLIPE)
drive_client = DriveClient.get()

# Nome do vídeo/imagem de fundo (o mesmo que o resto do pipeline gera) —
# vem do config.py, centralizado (config.NOME_VIDEO_BASE)
NOME_VIDEO_BASE = config.NOME_VIDEO_BASE

print(f"Vídeo: {NOME_ORACAO}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. ENVIAR O .ASS (o original do notebook anterior, ou uma       ║
# ║  versão que você corrigiu manualmente — tanto faz, contanto que  ║
# ║  seja um .ass válido)                                            ║
# ╚══════════════════════════════════════════════════════════════════╝
from google.colab import files

print("Selecione o arquivo .ass:")
enviados = files.upload()
nome_ass = list(enviados.keys())[0]
caminho_ass = Path(nome_ass)
print(f"✅ Recebido: {caminho_ass}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. BAIXAR O VÍDEO/IMAGEM DE FUNDO DO DRIVE                      ║
# ╚══════════════════════════════════════════════════════════════════╝
from srt_utils import ler_srt  # garante que os módulos do pipeline já foram testados no import

video_base_local = Path(NOME_VIDEO_BASE)
ok = drive_client.download(config.pasta_oracao, NOME_VIDEO_BASE, video_base_local)
if not ok:
    raise FileNotFoundError(f"Não achei '{NOME_VIDEO_BASE}' em {config.pasta_oracao} — ajuste NOME_VIDEO_BASE na célula 2")
print(f"✅ Vídeo base baixado: {video_base_local}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. QUEIMAR                                                      ║
# ╚══════════════════════════════════════════════════════════════════╝
from ffmpeg_utils import queimar_legendas_ass

# O .ass de 6 idiomas sai do caption-multicolor-zh-generate já com `_zh` no
# nome. Sem ler esse sufixo aqui, queimar a versão de 6 idiomas gravaria por
# cima do `_final_multicolor.mp4` de 5 -- mesmo nome, vídeo diferente, e nada
# avisaria: você só descobriria abrindo o arquivo. Por isso a variante do
# resultado vem do nome do arquivo que VOCÊ escolheu, não de uma opção que dá
# pra esquecer de marcar.
config.SUFIXO_VARIANTE_IDIOMAS = "_zh" if caminho_ass.stem.endswith("_zh") else ""
if config.SUFIXO_VARIANTE_IDIOMAS:
    print("🇨🇳 .ass de 6 idiomas detectado — o resultado vai com sufixo _zh")

caminho_saida = Path(config.NOME_VIDEO_FINAL_MULTICOLOR)
if caminho_saida.exists():
    print(f"⚠️  {caminho_saida.name} já existe aqui e vai ser refeito")
print(f"Saída: {caminho_saida.name}")

resultado = queimar_legendas_ass(
    video_entrada=video_base_local,
    ass_path=caminho_ass,
    saida=caminho_saida,
)
print(f"✅ Vídeo com legenda colorida: {resultado}")
print("\nRode a célula 6 pra baixar e conferir. Só rode a célula 7 (salvar no Drive) depois de confirmar que ficou bom.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  6. BAIXAR O RESULTADO (pra conferir)                            ║
# ╚══════════════════════════════════════════════════════════════════╝
files.download(str(resultado))


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  7. SALVAR NO DRIVE — só rode depois de conferir que ficou bom   ║
# ╚══════════════════════════════════════════════════════════════════╝
caminho_no_drive = drive_client.upload(resultado, config.pasta_oracao)
print(f"✅ Salvo no Drive: {caminho_no_drive}")
